<a href="https://colab.research.google.com/github/cintiapinho/Aulas_FATEC_PLN/blob/main/Materiais/Aula_5_6_ExtracaoCaracteristicas_ML_em_PLN/notebook_exercicios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sessões 05 e 06 — Exercícios (entrega)

**Processamento de Linguagem Natural**

Atividade prática

In [2]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

---

## Desafio — comparando os três métodos numa base bem mais suja

Até agora usamos o HateBR, que já vem bem preparado (uma coluna de texto, um rótulo limpo). Bases do mundo real raramente chegam assim. Vamos usar o **B2W-Reviews01** — mais de 130 mil avaliações reais de produtos da Americanas.com, coletadas em 2018, com nota de 1 a 5 estrelas e se a pessoa recomendaria o produto a um amigo (CC BY-NC-SA 4.0).

Fonte: [github.com/b2wdigital/b2w-reviews01](https://github.com/b2wdigital/b2w-reviews01)

Essa base é genuinamente suja: tem linha sem texto, review duplicada, texto em CAIXA ALTA, preço colado no meio da frase ("R$1994.20,eu consegui..."), números, pontuação estranha. Antes de classificar qualquer coisa, precisa limpar.

In [3]:
from sklearn.model_selection import train_test_split

url_b2w = "https://raw.githubusercontent.com/b2wdigital/b2w-reviews01/master/B2W-Reviews01.csv"
b2w = pd.read_csv(url_b2w, low_memory=False)

print("Linhas:", len(b2w))
print("Colunas:", b2w.columns.tolist())
print()
print("Nulos em review_text:", b2w["review_text"].isna().sum())
print("Nulos em recommend_to_a_friend:", b2w["recommend_to_a_friend"].isna().sum())
print("Reviews duplicadas:", b2w["review_text"].duplicated().sum())
print()
print(b2w[["overall_rating", "recommend_to_a_friend", "review_text"]].sample(3, random_state=1).to_string())

Linhas: 132373
Colunas: ['submission_date', 'reviewer_id', 'product_id', 'product_name', 'product_brand', 'site_category_lv1', 'site_category_lv2', 'review_title', 'overall_rating', 'recommend_to_a_friend', 'review_text', 'reviewer_birth_year', 'reviewer_gender', 'reviewer_state']

Nulos em review_text: 3275
Nulos em recommend_to_a_friend: 18
Reviews duplicadas: 5648

       overall_rating recommend_to_a_friend                                                                                                                       review_text
9967                1                    No                                                                                                                               NaN
62225               3                   Yes  a churrasqueira veio com defeito, o furo do meio não está centralizado e tem uma borda menor que a outra, mesmo assim me atende.
40902               2                    No                                                  Como vou saber

In [4]:
b2w

,submission_date,reviewer_id,product_id,product_name,product_brand,site_category_lv1,site_category_lv2,review_title,overall_rating,recommend_to_a_friend,review_text,reviewer_birth_year,reviewer_gender,reviewer_state
0,2018-01-01 00:11:28,d0fb1ca69422530334178f5c8624aa7a99da47907c44de...,132532965,Notebook Asus Vivobook Max X541NA-GO472T Intel...,NaN,Informática,Notebook,Bom,4,Yes,Estou contente com a compra entrega rápida o ú...,1958.0,F,RJ
1,2018-01-01 00:13:48,014d6dc5a10aed1ff1e6f349fb2b059a2d3de511c7538a...,22562178,Copo Acrílico Com Canudo 500ml Rocie,NaN,Utilidades Domésticas,"Copos, Taças e Canecas","Preço imbatível, ótima qualidade",4,Yes,"Por apenas R$1994.20,eu consegui comprar esse ...",1996.0,M,SC
2,2018-01-01 00:26:02,44f2c8edd93471926fff601274b8b2b5c4824e386ae4f2...,113022329,Panela de Pressão Elétrica Philips Walita Dail...,philips walita,Eletroportáteis,Panela Elétrica,ATENDE TODAS AS EXPECTATIVA.,4,Yes,SUPERA EM AGILIDADE E PRATICIDADE OUTRAS PANEL...,1984.0,M,SP
3,2018-01-01 00:35:54,ce741665c1764ab2d77539e18d0e4f66dde6213c9f0863...,113851581,Betoneira Columbus - Roma Brinquedos,roma jensen,Brinquedos,Veículos de Brinquedo,presente mais que desejado,4,Yes,MEU FILHO AMOU! PARECE DE VERDADE COM TANTOS D...,1985.0,F,SP
4,2018-01-01 01:00:28,7d7b6b18dda804a897359276cef0ca252f9932bf4b5c8e...,131788803,"Smart TV LED 43"" LG 43UJ6525 Ultra HD 4K com C...",lg,TV e Home Theater,TV,"Sem duvidas, excelente",5,Yes,"A entrega foi no prazo, as americanas estão de...",1994.0,M,MG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132368,2018-05-31 23:30:50,15f20e95ff44163f3175aaf67a5ae4a94d5030b409e521...,17962233,Carregador De Pilha Sony + 4 Pilhas Aa 2500mah,NaN,Câmeras e Filmadoras,Acessórios para Câmeras e Filmadoras,Ótimo produto!,5,Yes,"Vale muito, estou usando no controle do Xbox e...",1988.0,M,RS
132369,2018-05-31 23:42:25,def7cf9028b0673ab8bca3b1d06e085461fafb88cd48d9...,132631701,Mop Giratório Fit + Refil Extra - At Home,NaN,Utilidades Domésticas,Material de Limpeza,Sensacional,5,Yes,"Prático e barato, super indico o produto para ...",1979.0,F,SP
132370,2018-05-31 23:44:16,7bcbf542f5d7dd9a9a192a6805adba7a7a4c1ce3bf00df...,16095859,Fita Led 5m Rgb 3528 Siliconada Com 300 Leds C...,NaN,Automotivo,Iluminação,Ótimo produto,4,Yes,Chegou antes do prazo previsto e corresponde a...,1979.0,F,PR
132371,2018-05-31 23:46:48,e6fb0b19277d01c2a300c7837a105f3c369377e92f9c19...,6774907,Etiquetas Jurídicas Vade Mecum - Marca Fácil,marca facil,Papelaria,Material de Escritório,O produto não é bom.,1,No,"Material fraco, poderia ser melhor. Ficou deve...",1991.0,M,RJ


### Passo 1 — Limpar e preparar

O rótulo que vamos prever é `recommend_to_a_friend` ("Yes"/"No" — vira 1/0). Antes de vetorizar:

1. Tire as linhas com `review_text` ou `recommend_to_a_friend` nulos (`.dropna`).
2. Tire as reviews duplicadas (`.drop_duplicates(subset=["review_text"])`).
3. **(Sua vez)** Escreva uma função `limpar_review(texto)` usando regex que:
   - Converte pra minúsculas;
   - Troca valores em reais (`R$1994.20`, `r$ 50,00` etc.) por um marcador `" dinheiro "` (dica: `r'r\$\s*\d+[.,]?\d*'`);
   - Troca qualquer número restante por `" numero "`;
   - Remove tudo que não for letra (acentuada ou não) ou espaço;
   - Colapsa espaços duplos.
4. Aplique a função em toda a coluna `review_text` com `.apply(...)`.
5. Por que faz sentido trocar preço/número por um marcador em vez de simplesmente apagar? (Pense no que aconteceria com "R$1994.20,eu consegui comprar" se você só apagasse os dígitos sem colocar nada no lugar.)

Amostre **6.000 linhas** (`.sample(6000, random_state=1)`) antes de seguir — a base inteira é grande demais pra rodar tudo em segundos no Colab gratuito.

5. Por que faz sentido trocar preço/número por um marcador em vez de simplesmente apagar? (Pense no que aconteceria com "R$1994.20,eu consegui comprar" se você só apagasse os dígitos sem colocar nada no lugar.)

R:
Evitar colagem acidental de palavras: Se apagássemos apenas os números e pontuações de "R$1994.20,eu", o resultado viraria "reu", fundindo duas palavras distintas em um termo inexistente no vocabulário. O marcador com espaços em volta preserva a separação léxica.
Preservação de sinal semântico: Em análises de satisfação de e-commerce, o cliente mencionar preço (dinheiro) ou dados quantitativos (numero, ex: dias de entrega, medidas, voltagem) carrega forte peso opinativo (ex: "caro pelo dinheiro", "ótimo pelo dinheiro"). Substituir por tokens genéricos permite ao algoritmo capturar esse padrão sem explodir a dimensionalidade do vocabulário com infinitos números diferentes.


In [5]:
# 1. Remover nulos nas colunas essenciais e remover duplicadas
b2w_limpo = b2w.dropna(subset=["review_text", "recommend_to_a_friend"]).copy()
b2w_limpo = b2w_limpo.drop_duplicates(subset=["review_text"])

# 2. Amostrar 6.000 avaliações para garantir reprodutibilidade e velocidade
b2w_limpo = b2w_limpo.sample(6000, random_state=1).reset_index(drop=True)

# 3. Função de limpeza com Regex
def limpar_review(texto):
    # a. Converter para minúsculas
    texto = texto.lower()
    
    # b. Substituir valores em reais por ' dinheiro '
    # Dica da professora: r'r\$\s*\d+[.,]?\d*'
    texto = re.sub(r'r\$\s*\d+([.,]\d+)?', ' dinheiro ', texto)
    
    # c. Substituir qualquer outro número restante por ' numero '
    texto = re.sub(r'\d+', ' numero ', texto)
    
    # d. Remover tudo o que não for letra (incluindo acentuadas) ou espaço
    texto = re.sub(r'[^a-zà-ÿ\s]', ' ', texto)
    
    # e. Colapsar múltiplos espaços em um único
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

# 4. Aplicar a função na coluna review_text
b2w_limpo["texto_limpo"] = b2w_limpo["review_text"].apply(limpar_review)

# Visualizar o resultado (antes vs depois)
b2w_limpo[["review_text", "texto_limpo"]].head(3)


,review_text,texto_limpo
0,"Não gostei, Muito limitado Não reproduz as m...",não gostei muito limitado não reproduz as mídi...
1,"O produto é maravilhoso, resistente, de grande...",o produto é maravilhoso resistente de grande q...
2,"A qualidade superou as expectativas, e realmen...",a qualidade superou as expectativas e realment...


### Passo 2 — Os três métodos, na mesma base

Agora repita a comparação de hoje, só que nesta base nova:

1. **Detector por palavra-chave** (como `contem_linguagem_inadequada` das Sessões 03-04): monte uma lista pequena de palavras claramente negativas ("péssimo", "horrível", "não recomendo"...) e classifique como "não recomenda" (0) qualquer review que contenha alguma delas; o resto vira "recomenda" (1).
2. **Léxico de sentimento** (`polaridade_frase_simples` + `lexico`, já carregados no setup): some a polaridade da review; se a soma for negativa, classifique como "não recomenda".
3. **Classificador TF-IDF + Naive Bayes**: repita o loop de unigrama / uni+bigrama / uni+bi+trigrama que já fizemos no material, agora com `X_treino`/`X_teste` vindos de `texto_limpo`.

Pra cada um, calcule a acurácia contra `recommend_to_a_friend` (convertido pra 1/0) e monte uma tabela comparando os três, igual à do material de hoje.

**Antes de comparar:** calcule também a acurácia de um "modelo" que sempre chuta a classe mais frequente (`y_treino.mode()[0]` repetido pra todo mundo). Essa base **não é balanceada** como o HateBR — vale a pena saber qual é o piso antes de comemorar qualquer acurácia.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# ==============================================================================
# Preparação: Target binário (1 = Recomenda / Yes, 0 = Não Recomenda / No)
# ==============================================================================
y = (b2w_limpo["recommend_to_a_friend"] == "Yes").astype(int)
X = b2w_limpo["texto_limpo"]

# Divisão em Treino (80%) e Teste (20%) mantendo a proporção de classes (stratify)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Dicionário para guardar as acurácias e montar a tabela no final
resultados = {}

# ==============================================================================
# 2a. Baseline "Sempre a classe majoritária" (Piso)
# ==============================================================================
classe_majoritaria = y_treino.mode()[0]
pred_baseline = [classe_majoritaria] * len(y_teste)
acc_baseline = accuracy_score(y_teste, pred_baseline)
resultados["Baseline (classe majoritária)"] = acc_baseline
print(f"2a. Baseline (classe majoritária {classe_majoritaria}): {acc_baseline:.4f}")

# ==============================================================================
# 2b. Detector por palavra-chave negativa
# ==============================================================================
# Lista de termos fortemente associados a descontentamento e não recomendação
termos_negativos = [
    "pessimo", "péssimo", "horrivel", "horrível", "não recomendo", 
    "nao recomendo", "ruim", "defeito", "quebrado", "odiei", "lixo"
]

def classificar_palavra_chave(texto):
    # Se contiver alguma palavra negativa -> 0 (não recomenda), senão -> 1 (recomenda)
    texto_lower = str(texto).lower()
    return 0 if any(termo in texto_lower for termo in termos_negativos) else 1

pred_palavra = X_teste.apply(classificar_palavra_chave)
acc_palavra = accuracy_score(y_teste, pred_palavra)
resultados["Detector por Palavra-chave"] = acc_palavra
print(f"2b. Detector por palavra-chave: {acc_palavra:.4f}")

# ==============================================================================
# 2c. Léxico de Sentimento (OpLexicon)
# ==============================================================================
url_lexico = "https://raw.githubusercontent.com/marlovss/OpLexicon/main/OpLexicon.csv"
lexico_df = pd.read_csv(url_lexico, header=None, names=["palavra", "classe", "polaridade"])
lexico = dict(zip(lexico_df["palavra"], lexico_df["polaridade"]))

def polaridade_frase_simples(frase, lexico):
    palavras = re.findall(r"[a-zà-ÿA-ZÀ-ß]+", str(frase).lower())
    return sum(lexico.get(p, 0) for p in palavras)

# Se a soma da polaridade for negativa (< 0) -> 0 (não recomenda), senão -> 1 (recomenda)
pred_lexico = X_teste.apply(lambda t: 0 if polaridade_frase_simples(t, lexico) < 0 else 1)
acc_lexico = accuracy_score(y_teste, pred_lexico)
resultados["Léxico de Sentimento (OpLexicon)"] = acc_lexico
print(f"2c. Léxico de Sentimento: {acc_lexico:.4f}")

# ==============================================================================
# 2d. TF-IDF + Naive Bayes (Unigrama, Uni+Bigrama, Uni+Bi+Trigrama)
# ==============================================================================
cenarios_ml = [
    ("TF-IDF (unigrama) + Naive Bayes", (1, 1)),
    ("TF-IDF (uni+bigrama) + Naive Bayes", (1, 2)),
    ("TF-IDF (uni+bi+trigrama) + Naive Bayes", (1, 3))
]

for nome, ngrams in cenarios_ml:
    vetorizador = TfidfVectorizer(ngram_range=ngrams)
    Xtr = vetorizador.fit_transform(X_treino)
    Xte = vetorizador.transform(X_teste)
    
    modelo = MultinomialNB()
    modelo.fit(Xtr, y_treino)
    pred_nb = modelo.predict(Xte)
    
    acc_nb = accuracy_score(y_teste, pred_nb)
    resultados[nome] = acc_nb
    print(f"2d. {nome}: vocabulário={len(vetorizador.vocabulary_):>6} | acurácia={acc_nb:.4f}")

# ==============================================================================
# Tabela Comparativa Final
# ==============================================================================
df_comparativo = pd.DataFrame([
    {"Abordagem": k, "Acurácia": f"{v * 100:.2f}%"} 
    for k, v in resultados.items()
])
print("\n--- TABELA COMPARATIVA FINAL ---")
df_comparativo

2a. Baseline (classe majoritária 1): 0.7350
2b. Detector por palavra-chave: 0.7850
2c. Léxico de Sentimento: 0.7750
2d. TF-IDF (unigrama) + Naive Bayes: vocabulário=  8982 | acurácia=0.8175
2d. TF-IDF (uni+bigrama) + Naive Bayes: vocabulário= 58994 | acurácia=0.7842
2d. TF-IDF (uni+bi+trigrama) + Naive Bayes: vocabulário=133048 | acurácia=0.7717

--- TABELA COMPARATIVA FINAL ---


,Abordagem,Acurácia
0,Baseline (classe majoritária),73.50%
1,Detector por Palavra-chave,78.50%
2,Léxico de Sentimento (OpLexicon),77.50%
3,TF-IDF (unigrama) + Naive Bayes,81.75%
4,TF-IDF (uni+bigrama) + Naive Bayes,78.42%
5,TF-IDF (uni+bi+trigrama) + Naive Bayes,77.17%


### Passo 3 — Reflexão

---

#### 1. Qual dos três métodos ficou mais perto do teto (ML) e qual ficou mais perto do piso (baseline sempre-a-mesma-classe)?

* Piso (Baseline): 73,50%
* Teto (ML - TF-IDF Unigrama + Naive Bayes): 81,75%
* Detector por Palavras-chave: 78,08%
* Léxico de Sentimento (OpLexicon): 77,50%

O ML foi o TF-IDF (unigrama) + Naive Bayes com quase 82% de acurácia.

O Léxico de Sentimento ficou mais proximo do piso. O resultado baixo se da por conta do dicionario que é genérico e não reflete as girias/linguagem informal de compras online, além de somar palavras isoladas sem considerar negações.

Já o detector por palavras-chave foi que o que ficou mais proximo do ML, mas ambos métodos ficam "no meio do caminho", mostrando que listas fixas tem um um teto baixo comparadas a um modelo que aprende a distribuição estatística real das palavras.

---

#### 2. Essa base tem bem mais reviews positivas (recomenda) que negativas. Isso muda como você interpreta "86% de acurácia", por exemplo? Uma acurácia alta aqui significa a mesma coisa que uma acurácia alta no HateBR (que é balanceado 50/50)?

Muda pois a amostra está inviesada.

Como ~73,5% dos clientes recomendam o produto, se um modelo "burro" der sim pra todo mundo, ele já atinge 73,5 de acurácia sem precisar calcular nada. Por isso uma acurácia de ~82% ou até 86% nessa base representa um ganho significativo.

No HateBR, o piso da classe majoritária era de apenas 50%. Atingir 85% de acurácia lá representava um salto expressivo de +35 pontos percentuais.

---

#### 3. Se você tivesse que escolher só um desses três métodos pra colocar no ar amanhã, num sistema real da loja, qual escolheria — e o que pesaria mais na decisão além da acurácia (custo computacional, interpretabilidade, precisa de dado rotulado ou não)?

Escolheria o TF-IDF (Unigrama) + Naive Bayes, segue os motivos:

a. Primeiramnete pela acurácia em relação aos outros métodos que testamos aqui

b. Diferentemente de redes neurais profundas (BERT,LLMs), o Naive Bayes treinado com TF-IDF trás um custo computacional é bem baixo.

c. Bem mais fácil de entender o que está acontecendo, da pra por exemplo extrair exatamente quais palavras tem maiores probabilidades condicionais, isso permite que a equipe de negócio entenda o que está puxando as avaliações pra baixo por exemplo.